The script processes P-band Synthetic Aperture Radar (SAR) data by first setting up the directory paths and defining parameters like the heading direction and polarimetric channel. It loads various datasets including scene parameters, phase-to-height factors (kz), sensor trajectories, and digital terrain models (DTMs). Single Look Complex (SLC) images for the specified polarimetric channel are read and processed. The script performs repeat-pass tomography to generate and visualize tomographic profiles in both the azimuth and ground range directions. Additionally, it reads ancillary geocoding data from MetaSensing files and adjusts the data to match the scene parameters.

Comments on Changes or Uncertainties:
- Function Placeholder: The read_biomass_data function needs to be implemented based on the actual data format. This placeholder assumes it handles the reading of SLC and other data formats.
- Path Handling: Paths are constructed using os.path.join for better cross-platform compatibility.
- Polarimetric Index: Adjusted to 0-based indexing in Python as opposed to 1-based indexing in MATLAB.
- Scene Parameters: Loaded using scipy.io.loadmat to read .mat files.
- Tomographic Processing: Adjustments for handling complex numbers and numpy operations.
- Geocoding Data: Ancillary data from NetCDF files is read using the netCDF4 library, and indices are calculated to slice this data to match the scene parameters.

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from scipy.io import loadmat
from netCDF4 import Dataset

# Placeholder function to read biomass data (assumed to be implemented)
def read_biomass_data(filepath, param, mode):
    # Placeholder function
    # You need to implement this based on the actual data format
    pass

# Options

# Change directory to parent folder and set paths
os.chdir('..')
parent_path = os.path.join(os.getcwd(), 'SAR_data_Matlab', 'P')
os.chdir('Matlab_DEMO')

# Define heading and polarimetric channel
heading = 'NW'  # Either 'NW' (290°) or 'SE' (110°)
polarimetric_channel = 'HH'  # 'HH', 'HV' or 'VV'

# Determine the polarimetric index based on the channel
if polarimetric_channel == 'HH':
    pol_ind = 1
elif polarimetric_channel == 'HV':
    pol_ind = 2
elif polarimetric_channel == 'VV':
    pol_ind = 3

# Loading

# Loading scene parameters
curr_fullpath = os.path.join(parent_path, heading, 'Util', 'SLC', 'SceneParameters.mat')
scene_parameters = loadmat(curr_fullpath)

ground_range_axis = scene_parameters['ground_range_axis'].squeeze()
azimuth_axis = scene_parameters['azimuth_axis'].squeeze()
Ngr = len(ground_range_axis)
Naz = len(azimuth_axis)

# Loading phase-to-height factors
kz = np.zeros((Ngr, Naz, scene_parameters['N'][0][0]), dtype=np.single)
for n in range(scene_parameters['N'][0][0]):
    curr_fullpath = os.path.join(parent_path, heading, 'Util', 'SLC', 'kz', scene_parameters['filenames_kz'][n][0])
    print(f'Loading {curr_fullpath}...')
    kz[:, :, n] = read_biomass_data(curr_fullpath, [], 2)
    print('done.')

# Loading sensor trajectories
curr_fullpath = os.path.join(parent_path, heading, 'Util', 'SLC', 'FocusingCenters.mat')
focusing_centers = loadmat(curr_fullpath)

# Loading DTMs
curr_fullpath = os.path.join(parent_path, heading, 'Util', 'SLC', 'MetaSensingDTM.mat')
meta_sensing_dtm = loadmat(curr_fullpath)
curr_fullpath = os.path.join(parent_path, heading, 'Util', 'SLC', 'TomoDTMsr.mat')
tomo_dtm_sr = loadmat(curr_fullpath)

# Loading SLC images
I = np.zeros((Ngr, Naz, scene_parameters['N'][0][0]), dtype=np.single)
for n in range(scene_parameters['N'][0][0]):
    curr_fullpath = os.path.join(parent_path, heading, 'SLC', 'PoliMi', scene_parameters['filenames_slc'][n, pol_ind-1][0])
    print(f'Loading {curr_fullpath}...')
    I[:, :, n] = read_biomass_data(curr_fullpath, [], 1)
    print('done.')

# Tomography

# Local vertical axis (with respect to the local ground level)
z_axis = np.arange(-60, 61, 1)
Nz = len(z_axis)

# Azimuth profile
S = np.zeros((Nz, Naz))
gr_ind = round(Ngr / 4)
F = np.sum(np.abs(I[gr_ind, :, :])**2, axis=2)

for z_ind in range(Nz):
    z = z_axis[z_ind]

    # Ground steering
    Igs = I[gr_ind, :, :] * np.exp(-1j * kz[gr_ind, :, :] * (tomo_dtm_sr['TomoDTMsr'][gr_ind, :] - meta_sensing_dtm['MetaSensingDTM'][gr_ind, :]))

    S[z_ind, :] = np.abs(np.sum(Igs * np.exp(-1j * kz[gr_ind, :, :] * z), axis=2))**2 / (F + np.finfo(float).eps)

plt.figure()
plt.imshow(S, aspect='auto', extent=[azimuth_axis[0]*1e-3, azimuth_axis[-1]*1e-3, z_axis[-1], z_axis[0]], origin='lower')
plt.xlabel('Azimuth [km]')
plt.ylabel('[m]')
plt.title('Azimuth Profile')
plt.colorbar()
plt.show()

# Ground range profile
S = np.zeros((Nz, Ngr))
az_ind = round(Naz / 3)
F = np.sum(np.abs(I[:, az_ind, :])**2, axis=2)

for z_ind in range(Nz):
    z = z_axis[z_ind]

    # Ground steering
    Igs = I[:, az_ind, :] * np.exp(-1j * kz[:, az_ind, :] * (tomo_dtm_sr['TomoDTMsr'][:, az_ind] - meta_sensing_dtm['MetaSensingDTM'][:, az_ind]))

    S[z_ind, :] = np.abs(np.sum(Igs * np.exp(-1j * kz[:, az_ind, :] * z), axis=2))**2 / (F + np.finfo(float).eps)

plt.figure()
plt.imshow(S.T, aspect='auto', extent=[ground_range_axis[0]*1e-3, ground_range_axis[-1]*1e-3, z_axis[-1], z_axis[0]], origin='lower')
plt.xlabel('Ground range [km]')
plt.ylabel('[m]')
plt.title('Ground Range Profile')
plt.colorbar()
plt.show()

# Reading MetaSensing ancillary data
if heading == 'NW':
    curr_fullpath = os.path.join(parent_path, heading, 'SLC', 'MetaSensing', 'SAR_CPLX_20200722092915_435.0M_HH_12_pres_16_fdc_0.sar.sig.nc')
elif heading == 'SE':
    curr_fullpath = os.path.join(parent_path, heading, 'SLC', 'MetaSensing', 'SAR_CPLX_20200723082647_435.0M_HH_12_pres_16_fdc_0.sar.sig.nc')

with Dataset(curr_fullpath, 'r') as ds:
    LatImage_azimuth_axis = ds.variables['CrossRange'][:]
    LatImage_ground_range_axis = ds.variables['GroundRange'][:]
    LatImage = ds.variables['LatImage'][:]

# Determine indices for slicing the MetaSensing data to match the scene parameters
az_start_ind = np.argmin(np.abs(LatImage_azimuth_axis - azimuth_axis[0]))
az_stop_ind = np.argmin(np.abs(LatImage_azimuth_axis - azimuth_axis[-1]))
gr_start_ind = np.argmin(np.abs(LatImage_ground_range_axis - ground_range_axis[0]))
gr_stop_ind = np.argmin(np.abs(LatImage_ground_range_axis - ground_range_axis[-1]))

LatImage = LatImage[gr_start_ind:gr_stop_ind+1, az_start_ind:az_stop_ind+1]
